In [1]:
import numpy as np
import pandas as pd
import random

In [18]:
# Important landmarks for Warrior II pose
IMPORTANT_LMS = [
    "nose",             # 0
    "left_ear",         # 7
    "right_ear",        # 8
    "left_shoulder",    # 11
    "right_shoulder",   # 12
    "left_elbow",       # 13
    "right_elbow",      # 14
    "left_wrist",       # 15
    "right_wrist",      # 16
    "left_index",       # 19
    "right_index",      # 20
    "left_hip",         # 23
    "right_hip",        # 24
    "left_knee",        # 25
    "right_knee",       # 26
    "left_ankle",       # 27
    "right_ankle",      # 28
    "left_heel",        # 29
    "right_heel"        # 30
]



# Important Functions

In [22]:


def get_landmark_columns(landmark_name):
    """Get all column names for a landmark (x, y, z, visibility)"""
    return [
        f'{landmark_name}_x',
        f'{landmark_name}_y',
        f'{landmark_name}_z',
        f'{landmark_name}_v'
    ]

def modify_landmark(landmarks, landmark_name, x_delta=0, y_delta=0, z_delta=0):
    """Helper function to modify a specific landmark"""
    modified = landmarks.copy()
    modified[f'{landmark_name}_x'] += x_delta
    modified[f'{landmark_name}_y'] += y_delta
    modified[f'{landmark_name}_z'] += z_delta
    return modified

# --- Incorrect pose variations for Mountain Pose ---

def create_feet_too_close(landmarks):
    """Feet too close together instead of together or slightly apart"""
    modified = landmarks.copy()
    move_amount = np.random.uniform(0.05, 0.15)
    
    # Move heels closer together
    modified = modify_landmark(modified, 'left_heel', x_delta=move_amount/2)
    modified = modify_landmark(modified, 'right_heel', x_delta=-move_amount/2)
    
    # Slightly adjust ankles for realism
    modified = modify_landmark(modified, 'left_ankle', x_delta=move_amount/3)
    modified = modify_landmark(modified, 'right_ankle', x_delta=-move_amount/3)
    
    return modified


def create_arms_dropped(landmarks):
    """Arms hanging too low instead of relaxed by sides"""
    modified = landmarks.copy()
    drop_amount = np.random.uniform(0.1, 0.25)
    modified = modify_landmark(modified, 'left_elbow', y_delta=drop_amount)
    modified = modify_landmark(modified, 'left_index', y_delta=drop_amount)
    modified = modify_landmark(modified, 'right_elbow', y_delta=drop_amount)
    modified = modify_landmark(modified, 'right_index', y_delta=drop_amount)
    return modified

def create_shoulders_raised(landmarks):
    """Shoulders raised/tensed instead of relaxed"""
    modified = landmarks.copy()
    raise_amount = np.random.uniform(0.05, 0.12)
    modified = modify_landmark(modified, 'left_shoulder', y_delta=-raise_amount)
    modified = modify_landmark(modified, 'right_shoulder', y_delta=-raise_amount)
    return modified

def create_leaning_forward(landmarks):
    """Torso leaning forward instead of upright"""
    modified = landmarks.copy()
    lean_amount = np.random.uniform(0.05, 0.15)
    forward_shift = np.random.uniform(0.02, 0.06)
    
    # Upper body leans forward
    for lm in ['nose', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder']:
        modified = modify_landmark(modified, lm, y_delta=lean_amount, x_delta=forward_shift)
    
    return modified

def create_leaning_back(landmarks):
    """Torso leaning backward instead of upright"""
    modified = landmarks.copy()
    lean_amount = np.random.uniform(0.05, 0.15)
    backward_shift = np.random.uniform(0.02, 0.06)
    
    # Upper body leans backward
    for lm in ['nose', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder']:
        modified = modify_landmark(modified, lm, y_delta=-lean_amount, x_delta=-backward_shift)
    
    return modified


def create_head_tilted(landmarks):
    """Head tilted instead of neutral"""
    modified = landmarks.copy()
    tilt_amount = np.random.uniform(0.03, 0.1)
    
    if random.random() < 0.5:
        # Tilt left
        modified = modify_landmark(modified, 'nose', x_delta=-tilt_amount)
        modified = modify_landmark(modified, 'left_ear', x_delta=-tilt_amount/2)
        modified = modify_landmark(modified, 'right_ear', x_delta=tilt_amount/2)
    else:
        # Tilt right
        modified = modify_landmark(modified, 'nose', x_delta=tilt_amount)
        modified = modify_landmark(modified, 'left_ear', x_delta=-tilt_amount/2)
        modified = modify_landmark(modified, 'right_ear', x_delta=tilt_amount/2)
    
    return modified



# Main function to create augumented dataset

In [23]:
import pandas as pd

def augment_mountain_dataset(input_csv, output_csv, samples_per_class=50):
    """
    Create augmented dataset with multiple incorrect Mountain Pose classes
    Works specifically with 19 important landmarks

    Args:
        input_csv: Path to CSV with correct Mountain poses (filtered to 19 landmarks)
        output_csv: Path to save augmented dataset
        samples_per_class: Number of samples to generate for each incorrect class
    """
    print(f"📊 Loading correct poses from {input_csv}...")
    df = pd.read_csv(input_csv)

    # Verify we have the right landmarks
    expected_cols = ['label']
    for lm in IMPORTANT_LMS:
        expected_cols.extend(get_landmark_columns(lm.lower()))


    missing_cols = [col for col in expected_cols if col not in df.columns]
    if missing_cols:
        print(f"⚠️  Warning: Missing columns: {missing_cols[:5]}...")
        print(f"   Make sure your CSV has been filtered to the 19 important landmarks")

    # Filter only correct poses
    df_correct = df[df['label'] == 'c'].copy()
    print(f"✅ Found {len(df_correct)} correct pose samples")
    print(f"📏 Using {len([col for col in df.columns if col != 'label']) // 4} landmarks")

    # Define Mountain Pose-specific augmentation functions with their labels
    augmentations = {
        'feet_too_close': create_feet_too_close,
        'arms_dropped': create_arms_dropped,
        'shoulders_raised': create_shoulders_raised,
        'leaning_forward': create_leaning_forward,
        'leaning_back': create_leaning_back,
        'head_tilted': create_head_tilted
    }

    # Store all augmented samples
    all_samples = [df_correct]

    # Generate augmented samples for each class
    for label, aug_func in augmentations.items():
        print(f"🔄 Generating {samples_per_class} samples for class: {label}")
        augmented_samples = []

        for _ in range(samples_per_class):
            # Randomly select a correct pose
            correct_sample = df_correct.sample(n=1).iloc[0].copy()

            # Apply augmentation (exclude 'label' column)
            landmark_cols = [col for col in correct_sample.index if col != 'label']
            landmarks = correct_sample[landmark_cols]

            # Apply the augmentation function
            modified_landmarks = aug_func(landmarks)

            # Add label
            modified_landmarks['label'] = label

            augmented_samples.append(modified_landmarks)

        # Convert to dataframe
        df_aug = pd.DataFrame(augmented_samples)
        all_samples.append(df_aug)
        print(f"   ✓ Created {len(df_aug)} samples")

    # Combine all samples
    df_final = pd.concat(all_samples, ignore_index=True)

    # Shuffle the dataset
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

    # Save to CSV
    df_final.to_csv(output_csv, index=False)

    print(f"\n✅ Augmented dataset saved to {output_csv}")
    print(f"\n📈 Class Distribution:")
    class_counts = df_final['label'].value_counts().sort_index()
    for label, count in class_counts.items():
        print(f"   {label:25s}: {count:4d} samples")
    print(f"\n{'='*50}")
    print(f"Total samples: {len(df_final)}")
    print(f"Total classes: {df_final['label'].nunique()}")
    print(f"Features per sample: {len([col for col in df_final.columns if col != 'label'])}")

    return df_final


In [ ]:
# df = pd.read_csv("data_csv/train.csv")
# print(len(df))

420


In [24]:
if __name__ == "__main__":
    # Generate augmented dataset
    df = pd.read_csv("train.csv")

    no_of_data_per_class = len(df)
    df_augmented = augment_mountain_dataset(
        input_csv='train.csv',      # filtered CSV 
        output_csv='train_augmented.csv',    # Output with all classes
        samples_per_class = no_of_data_per_class                  # Adjust based on your needs
    )
        
    print("\n🎯 Dataset ready for training!")

📊 Loading correct poses from train.csv...
✅ Found 1083 correct pose samples
📏 Using 19 landmarks
🔄 Generating 1083 samples for class: feet_too_close
   ✓ Created 1083 samples
🔄 Generating 1083 samples for class: arms_dropped
   ✓ Created 1083 samples
🔄 Generating 1083 samples for class: shoulders_raised
   ✓ Created 1083 samples
🔄 Generating 1083 samples for class: leaning_forward
   ✓ Created 1083 samples
🔄 Generating 1083 samples for class: leaning_back
   ✓ Created 1083 samples
🔄 Generating 1083 samples for class: head_tilted
   ✓ Created 1083 samples

✅ Augmented dataset saved to train_augmented.csv

📈 Class Distribution:
   arms_dropped             : 1083 samples
   c                        : 1083 samples
   feet_too_close           : 1083 samples
   head_tilted              : 1083 samples
   leaning_back             : 1083 samples
   leaning_forward          : 1083 samples
   shoulders_raised         : 1083 samples

Total samples: 7581
Total classes: 7
Features per sample: 76

🎯 